In [ ]:
# =========================
# nb_reports
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

from pyspark.sql import functions as F
import json

output_base = f"/lakehouse/default/Files/output/period={period}/run_id={run_id}"

# ensure folder exists (Fabric will create on write, but mkdirs is explicit)
try:
    mssparkutils.fs.mkdirs(output_base)
except Exception:
    pass

# ---------- Load run data ----------
ch = spark.table("canonical_holdings").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)
ve = spark.table("validation_events").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)
pa = spark.table("policy_aggregates").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if ch.rdd.isEmpty():
    print("[nb_reports] No canonical data for this run.")
    mssparkutils.notebook.exit("NO_DATA")

dfms = [r["dfm_id"] for r in ch.select("dfm_id").distinct().collect()]

# ---------- Report 1 per DFM ----------
# validation failures + not_evaluable by policy + rule
for d in dfms:
    r1 = (
        ve.filter(F.col("dfm_id") == d)
          .groupBy("dfm_id", "policy_id", "rule_id", "severity", "status")
          .agg(F.count("*").alias("event_count"))
          .orderBy("policy_id", "rule_id", "status")
    )

    out = f"{output_base}/report1_{d}.csv"
    r1.coalesce(1).write.mode("overwrite").option("header", True).csv(out)

# ---------- Report 2 rollup ----------
r2 = (
    ve.groupBy("dfm_id", "rule_id", "severity", "status")
      .agg(F.count("*").alias("event_count"))
      .orderBy("dfm_id", "rule_id", "status")
)

r2.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{output_base}/report2_rollup.csv")

# ---------- reconciliation_summary.json ----------
# DFM-level totals from policy_aggregates + row counts from canonical_holdings
totals = (
    pa.groupBy("dfm_id")
      .agg(
          F.sum("total_bid_value_gbp").cast("double").alias("total_bid_value_gbp"),
          F.sum("total_cash_value_gbp").cast("double").alias("total_cash_value_gbp"),
          F.sum("total_accrued_interest_gbp").cast("double").alias("total_accrued_interest_gbp"),
          F.countDistinct("policy_id").alias("policy_count")
      )
)

rows = (
    ch.groupBy("dfm_id")
      .agg(F.count("*").alias("holding_rows"))
)

summary_df = totals.join(rows, "dfm_id", "full").fillna(0)

summary_records = [r.asDict() for r in summary_df.collect()]

payload = {
    "period": period,
    "run_id": run_id,
    "generated_at_utc": None,  # optional, can be set in notebook with current timestamp string
    "dfm_summaries": summary_records
}

json_path = f"{output_base}/reconciliation_summary.json"
mssparkutils.fs.put(json_path, json.dumps(payload, indent=2), True)

print(f"[nb_reports] Wrote outputs under: {output_base}")
mssparkutils.notebook.exit("OK")